In [1]:
# Import packages
from langchain.messages import AIMessage
from langchain_core.output_parsers import PydanticOutputParser
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage
import os
from langchain.tools import tool, ToolRuntime
import time
from dotenv import load_dotenv


In [2]:
load_dotenv()

# Connect notebook to langsmith
os.environ['LANGSMITH_TRACING'] = os.getenv('LANGSMITH_TRACING')
os.environ['LANGSMITH_ENDPOINT'] = os.getenv('LANGSMITH_ENDPOINT')
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['LANGSMITH_PROJECT'] = os.getenv('LANGSMITH_PROJECT')
os.environ['DEEPSEEK_API_KEY'] = os.getenv('DEEPSEEK_API_KEY')

- Simplest way to create a tool is with the @tool decorator.

- The function's docstring becomes the tool's description that helps the model understand when to use it.

- Type hints (def function(expression:str) -> str) **must** be provided as they define the tool's input schema.

In [3]:
# Creating a basic tool
# Modify the answer by removing the caps lock text.
# RETURN THE ANSWER AS A SINGLE VALUE WITH NO ADDITIONAL TEXT
@tool("multiplication_calculator") # -> Defines tool name
def multiplication_calculator(expression:str, runtime:ToolRuntime)->str:
  """Evaluate a mathematical expression and return the result.

  Args:
    expresion: A mathematical expression like:
       - "y by z", or
       - "y*z", or
       - "y multiplied by z", or
       - "y times z", or
       - "y x z"

    In which case, y and z are both numbers, whether floats or integers.

    Before entering the expression into the function, make sure that this rewritten, if necessary, into the format y*z.



  Returns:
    Dictionary with 'result' or 'error'
  """

  # Add logs execution identity information
  info = runtime.execution_info
  print(f"Thread: {info.thread_id}, Run: {info.run_id}")
  print(f"Attempt: {info.node_attempt}")

  try:
    # Evaluate the expression
    result = eval(expression)
    return {
        "success'": True,
        "result": result,
        "expression": expression
    }
  except Exception as e:
    return {
        "success": False,
        "error": str(e),
        "expression":expression
    }

In [4]:
ai_context = """You are a helpful assistant"""

human_message = """What is 12836458 times 192865?"""


In [5]:
agent = create_agent(model = "deepseek:deepseek-v4-flash", tools=[multiplication_calculator])
result = agent.invoke(
        {"messages": [
            {"role": "user",
             "content": f"{human_message}"}]},
          context=ai_context)

/Users/davidjeon/Documents/dsr_materials/dsr_agentic_ai/.venv/lib/python3.14/site-packages/pydantic/functional_validators.py:851: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value='You are a helpful assistant', input_type=str])
  function=lambda v, h: h(v), schema=original_schema
/Users/davidjeon/Documents/dsr_materials/dsr_agentic_ai/.venv/lib/python3.14/site-packages/pydantic/main.py:520: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value='You are a helpful assistant', input_type=str])
  return self.__pydantic_serializer__.to_python(


Thread: None, Run: None
Attempt: 1


In [6]:
print(result)

{'messages': [HumanMessage(content='What is 12836458 times 192865?', additional_kwargs={}, response_metadata={}, id='cf91ca81-4819-4947-8e25-67adc40f8c2d'), AIMessage(content='', additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer multiplication. Need use tool. Expression 12836458*192865.'}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 426, 'total_tokens': 490, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 18, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 426}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '14cc9b3f-b397-4866-bbee-a3595730d96d', 'finish_reason': 'tool_calls', 'logprobs': 

In [7]:
ai_messages = [
    m for m in result["messages"]
    if isinstance(m, AIMessage)
]
#print(ai_messages)
output_list=[message.content for message in ai_messages]
output_list

['', '12836458 × 192865 = **2,475,703,472,170**']

In [9]:
# If you want a neater output
print(output_list[1])

12836458 × 192865 = **2,475,703,472,170**


In [11]:
# Let's try something more interesting
# Let's try something more interesting
from bs4 import BeautifulSoup
import re
import urllib3

@tool("weather_page")
def get_weather(expression: str) -> str:
  """Gets the current weather for Berlin by scraping weather data from HTML response.

  Args:
    expression: The user's weather query (e.g., "What's the weather in Berlin?")

  Returns:
    A string containing the parsed weather information or an error message
  """

  try:
      # Create a PoolManager instance for sending requests
      http = urllib3.PoolManager()
      url = 'https://www.theweathernetwork.com/en/city/de/berlin/berlin/current'

      # Send GET request
      resp = http.request("GET", url)

      # Parse HTML with BeautifulSoup
      soup = BeautifulSoup(resp.data, 'html.parser')

      return soup

  except Exception as e:
      return f"Error fetching weather: {str(e)}"

In [12]:

ai_context = """You are a helpful assistant"""

human_message = """What's the weather in Berlin like?"""

agent = create_agent(model = "deepseek:deepseek-v4-flash", tools=[get_weather])
result = agent.invoke(
        {"messages": [
            {"role": "user",
             "content": f"{human_message}"}]},
          context=ai_context)

In [ ]:
ai_messages = [
    m for m in result["messages"]
    if isinstance(m, AIMessage)
]
#print(ai_messages)
output_list=[message.content for message in ai_messages]
for i in range(len(output_list)):
    print(outpit_list[i])

["I'll check the current weather in Berlin for you.",
 "Here's the current weather in Berlin (from The Weather Network, updated a few minutes ago):\n\n**Now: 2°C — Partly cloudy**\n- Feels like: −2°C\n- Today's high/low: 3°C / 0°C\n- Wind: 16 km/h from the NW\n- Humidity: 75% (dew point −2°C)\n- Pressure: 101.8 kPa and rising\n- Visibility: 11 km\n- UV index: Low\n- No active weather alerts\n\n**Short-term outlook:** A light rain-snow mix is expected early this afternoon (40–60% chance of precipitation, under 1 mm), followed by cloudy conditions with clear breaks into the evening. Temperatures hover around 2°C, feeling closer to −4°C as the afternoon wears on.\n\n**Sun:** Sunrise 7:41 am, sunset 4:04 pm — a short daylight window, so plan accordingly if you're heading out.\n\nBottom line: cold, damp, and breezy with a bit of wintry mix possible — worth a warm coat, and watch for slick spots on sidewalks and side streets this afternoon."]

In [ ]:

ai_context = """You are a helpful assistant"""

human_message = """What's the weather in Berlin like? Get it in celcius and return it to me in Fahrenheit by multiplying by 1.8 and adding the outcome to 32"""

agent = create_agent(model = "deepseek:deepseek-v4-flash", tools=[get_weather, multiplication_calculator])
result = agent.invoke(
        {"messages": [
            {"role": "user",
             "content": f"{human_message}"}]},
          context=ai_context)